# NAN VALUES REMOVAL


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tensorpac import EventRelatedPac

import os
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.paths import ERPAC_DIR, ROI_STCS_DIR
from config.config import COUPLINGS, TASK_STAGES, ROI, TOI, GROUPS, SUBJECTS

%matplotlib qt

CHECK FOR NANs IN STCS

In [ ]:
# ============================================================
# 1. LOAD TIME-RESOLVED ERPAC DATA
# ============================================================

erpac_df = pd.read_parquet(
    os.path.join(
        ERPAC_DIR,
        "erpac_results_nans.parquet"
    )
)

In [ ]:
erpac_df[erpac_df["sub"] == "s1_pac_sub34"] # has nan values

In [ ]:
n_nan = erpac_df["erpac_value"].isna().sum()
n_total = len(erpac_df)

print("NaNs:", n_nan)
print("Total:", n_total)
print("Percent NaN:", 100 * n_nan / n_total)

NaNs: 233685
Total: 14644260
Percent NaN: 1.5957446808510638


In [ ]:
missing_by_sub = (
    erpac_df
    .groupby("sub")["erpac_value"]
    .agg(
        n_total="size",
        n_nan=lambda x: x.isna().sum()
    )
)

missing_by_sub["pct_nan"] = (
    100 * missing_by_sub["n_nan"] /
    missing_by_sub["n_total"]
)

print(missing_by_sub.sort_values("pct_nan", ascending=False))

              n_total   n_nan  pct_nan
sub                                   
s1_pac_sub34   311580  155790     50.0
s1_pac_sub22   311580   77895     25.0
s1_pac_sub07   311580       0      0.0
s1_pac_sub11   311580       0      0.0
s1_pac_sub12   311580       0      0.0
s1_pac_sub14   311580       0      0.0
s1_pac_sub01   311580       0      0.0
s1_pac_sub15   311580       0      0.0
s1_pac_sub17   311580       0      0.0
s1_pac_sub20   311580       0      0.0
s1_pac_sub21   311580       0      0.0
s1_pac_sub23   311580       0      0.0
s1_pac_sub24   311580       0      0.0
s1_pac_sub26   311580       0      0.0
s1_pac_sub10   311580       0      0.0
s1_pac_sub29   311580       0      0.0
s1_pac_sub32   311580       0      0.0
s1_pac_sub33   311580       0      0.0
s1_pac_sub35   311580       0      0.0
s1_pac_sub36   311580       0      0.0
s1_pac_sub38   311580       0      0.0
s1_pac_sub40   311580       0      0.0
s1_pac_sub41   311580       0      0.0
s1_pac_sub43   311580    

In [ ]:
summary = (
    erpac_df
    .groupby(
        ["sub", "task", "task_stage"]
    )["erpac_value"]
    .agg(
        n="size",
        n_nan=lambda x: x.isna().sum(),
        n_valid=lambda x: x.notna().sum()
    )
    .reset_index()
)

summary["pct_nan"] = (
    summary["n_nan"] / summary["n"] * 100
)

summary[
    summary["sub"].isin(
        ["s1_pac_sub22", "s1_pac_sub34"]
    )
]

,sub,task,task_stage,n,n_nan,n_valid,pct_nan
20,s1_pac_sub22,FTT,go,176040,44010,132030,25.0
21,s1_pac_sub22,FTT,plan,135540,33885,101655,25.0
34,s1_pac_sub34,FTT,go,176040,88020,88020,50.0
35,s1_pac_sub34,FTT,plan,135540,67770,67770,50.0


LOCATE AND REMOVE VERTICES PRODUCING NAN VALUES

In [ ]:
# ============================================================
# CONDITIONS TO RECHECK
# ============================================================

CHECK_CONDITIONS = [
    {
        "group": "Y",
        "sub": "s1_pac_sub22",
        "task": "FTT",
        "stage": "go",
    },
    {
        "group": "Y",
        "sub": "s1_pac_sub22",
        "task": "FTT",
        "stage": "plan",
    },
    {
        "group": "O",
        "sub": "s1_pac_sub34",
        "task": "FTT",
        "stage": "go",
    },
    {
        "group": "O",
        "sub": "s1_pac_sub34",
        "task": "FTT",
        "stage": "plan",
    },
]


# ============================================================
# OUTPUT CONTAINERS
# ============================================================

erpac_nan_rows = []
erpac_label_summary = []
rerun_erpac_results = {}


# ============================================================
# LOOP THROUGH AFFECTED CONDITIONS
# ============================================================

for item in CHECK_CONDITIONS:

    group = item["group"]
    sub = item["sub"]
    task = item["task"]
    stage = item["stage"]

    print("\n" + "=" * 90)
    print(f"PROCESSING: {group} | {sub} | {task} | {stage}")
    print("=" * 90)

    # --------------------------------------------------------
    # LOAD SAVED ROI SOURCE DATA
    # --------------------------------------------------------

    roi_path = os.path.join(
        ROI_STCS_DIR,
        group,
        task,
        stage,
        f"{sub}_{task}_{stage}_roi_stcs.pkl",
    )

    print("Loading:")
    print(roi_path)

    with open(roi_path, "rb") as f:
        roi_data = pickle.load(f)

    sf = roi_data["M1"]["G_precentral-lh"]["sfreq"]

    condition_key = (
        group,
        sub,
        task,
        stage,
    )

    rerun_erpac_results[condition_key] = {}

    # ========================================================
    # COUPLING LOOP
    # ========================================================

    for coupling_name, (phase_freq, amp_freq) in COUPLINGS.items():

        print("\n" + "-" * 80)
        print(f"Coupling: {coupling_name}")
        print("-" * 80)

        p = EventRelatedPac(
            f_pha=phase_freq,
            f_amp=amp_freq,
        )

        amp_freqs = p.yvec

        rerun_erpac_results[condition_key][coupling_name] = {}

        # ====================================================
        # ROI LOOP
        # ====================================================

        for roi, labels in roi_data.items():

            print(f"\nROI: {roi}")

            rerun_erpac_results[
                condition_key
            ][coupling_name][roi] = {}

            # =================================================
            # LABEL LOOP
            # =================================================

            for label_name, label_data in labels.items():

                data = np.asarray(
                    label_data["data"]
                )

                # epochs x vertices x times
                n_epochs, n_vertices, n_times = data.shape

                n_freqs = len(amp_freqs)

                print(
                    f"  {label_name}: "
                    f"{n_epochs} epochs, "
                    f"{n_vertices} vertices, "
                    f"{n_times} times"
                )

                vertex_erpac = np.full(
                    (
                        n_vertices,
                        n_freqs,
                        n_times,
                    ),
                    np.nan,
                )

                vertex_pvalues = np.full(
                    (
                        n_vertices,
                        n_freqs,
                        n_times,
                    ),
                    np.nan,
                )

                # label counters
                bad_vertices = 0
                fully_nan_vertices = 0
                total_nan = 0

                # =============================================
                # VERTEX LOOP
                # =============================================

                for v in range(n_vertices):

                    x = data[:, v, :]

                    # -----------------------------------------
                    # Sanity check before ERPAC
                    # -----------------------------------------

                    source_has_nan = np.isnan(x).any()
                    source_has_inf = np.isinf(x).any()

                    if source_has_nan or source_has_inf:

                        print(
                            f"    WARNING source problem: "
                            f"vertex {v}"
                        )

                    # -----------------------------------------
                    # ERPAC
                    # -----------------------------------------

                    erpac = p.filterfit(
                        sf,
                        x,
                        method="circular",
                        mcp="bonferroni",
                        n_jobs=-1,
                    ).squeeze()

                    pvalues = p.pvalues.squeeze()

                    # -----------------------------------------
                    # STORE
                    # -----------------------------------------

                    vertex_erpac[v] = erpac
                    vertex_pvalues[v] = pvalues

                    # -----------------------------------------
                    # CHECK ERPAC FOR NaNs
                    # -----------------------------------------

                    nan_mask = np.isnan(erpac)

                    n_nan = nan_mask.sum()
                    n_total = erpac.size

                    pct_nan = (
                        100 * n_nan / n_total
                    )

                    if n_nan > 0:

                        bad_vertices += 1
                        total_nan += n_nan

                        if n_nan == n_total:
                            fully_nan_vertices += 1

                        # identify affected frequency/time bins
                        nan_freq_idx, nan_time_idx = np.where(
                            nan_mask
                        )

                        erpac_nan_rows.append({
                            "group": group,
                            "sub": sub,
                            "task": task,
                            "stage": stage,
                            "coupling": coupling_name,
                            "roi": roi,
                            "label": label_name,

                            "vertex_index": v,
                            "vertex_number":
                                label_data["vertices"][v],

                            "n_epochs": n_epochs,

                            "n_nan": n_nan,
                            "n_total": n_total,
                            "pct_nan": pct_nan,

                            "fully_nan_vertex":
                                n_nan == n_total,

                            "n_nan_freqs":
                                len(
                                    np.unique(
                                        nan_freq_idx
                                    )
                                ),

                            "n_nan_times":
                                len(
                                    np.unique(
                                        nan_time_idx
                                    )
                                ),

                            "source_nan":
                                source_has_nan,

                            "source_inf":
                                source_has_inf,
                        })

                        print(
                            f"    NaN ERPAC: vertex {v} "
                            f"(FS vertex "
                            f"{label_data['vertices'][v]}) | "
                            f"{n_nan}/{n_total} "
                            f"({pct_nan:.2f}%)"
                        )

                # =============================================
                # LABEL SUMMARY
                # =============================================

                label_n_total = vertex_erpac.size
                label_n_nan = np.isnan(
                    vertex_erpac
                ).sum()

                label_pct_nan = (
                    100
                    * label_n_nan
                    / label_n_total
                )

                erpac_label_summary.append({
                    "group": group,
                    "sub": sub,
                    "task": task,
                    "stage": stage,
                    "coupling": coupling_name,
                    "roi": roi,
                    "label": label_name,

                    "n_epochs": n_epochs,
                    "n_vertices": n_vertices,

                    "bad_vertices":
                        bad_vertices,

                    "fully_nan_vertices":
                        fully_nan_vertices,

                    "n_nan":
                        label_n_nan,

                    "n_total":
                        label_n_total,

                    "pct_nan":
                        label_pct_nan,
                })

                print(
                    f"  Summary: "
                    f"{bad_vertices}/{n_vertices} "
                    f"vertices contain NaNs | "
                    f"{label_pct_nan:.2f}% "
                    f"of ERPAC array NaN"
                )

                # =============================================
                # SAVE ERPAC RESULT
                # =============================================

                rerun_erpac_results[
                    condition_key
                ][coupling_name][roi][label_name] = {
                    "erpac":
                        vertex_erpac,

                    "p-values":
                        vertex_pvalues,

                    "times":
                        label_data["times"],

                    "vertices":
                        label_data["vertices"],

                    "amp_freqs":
                        amp_freqs,
                }


# ============================================================
# CREATE DIAGNOSTIC DATAFRAMES
# ============================================================

erpac_nan_df = pd.DataFrame(
    erpac_nan_rows
)

erpac_label_summary_df = pd.DataFrame(
    erpac_label_summary
)


# ============================================================
# PRINT LABEL-LEVEL SUMMARY
# ============================================================

print("\n\n")
print("=" * 100)
print("ERPAC NaN SUMMARY BY LABEL")
print("=" * 100)

print(
    erpac_label_summary_df[
        erpac_label_summary_df["n_nan"] > 0
    ].to_string(
        index=False
    )
)


# ============================================================
# ROI-LEVEL SUMMARY
# ============================================================

roi_summary = (
    erpac_label_summary_df
    .groupby(
        [
            "group",
            "sub",
            "task",
            "stage",
            "coupling",
            "roi",
        ],
        as_index=False,
    )
    .agg(
        n_vertices=(
            "n_vertices",
            "sum",
        ),

        bad_vertices=(
            "bad_vertices",
            "sum",
        ),

        fully_nan_vertices=(
            "fully_nan_vertices",
            "sum",
        ),

        n_nan=(
            "n_nan",
            "sum",
        ),

        n_total=(
            "n_total",
            "sum",
        ),
    )
)

roi_summary["pct_nan"] = (
    100
    * roi_summary["n_nan"]
    / roi_summary["n_total"]
)


print("\n")
print("=" * 100)
print("ERPAC NaN SUMMARY BY ROI")
print("=" * 100)

print(
    roi_summary[
        roi_summary["n_nan"] > 0
    ].to_string(
        index=False
    )
)


# ============================================================
# CONDITION / COUPLING SUMMARY
# ============================================================

condition_summary = (
    roi_summary
    .groupby(
        [
            "group",
            "sub",
            "task",
            "stage",
            "coupling",
        ],
        as_index=False,
    )
    .agg(
        n_nan=(
            "n_nan",
            "sum",
        ),

        n_total=(
            "n_total",
            "sum",
        ),

        bad_vertices=(
            "bad_vertices",
            "sum",
        ),

        fully_nan_vertices=(
            "fully_nan_vertices",
            "sum",
        ),
    )
)

condition_summary["pct_nan"] = (
    100
    * condition_summary["n_nan"]
    / condition_summary["n_total"]
)


print("\n")
print("=" * 100)
print("ERPAC NaN SUMMARY BY CONDITION")
print("=" * 100)

print(
    condition_summary.to_string(
        index=False
    )
)


In [39]:
# ============================================================
# CONDITIONS TO REPLACE
# ============================================================

AFFECTED_CONDITIONS = [
    {
        "group": "Y",
        "sub": "s1_pac_sub22",
        "task": "FTT",
        "stage": "go",
    },
    {
        "group": "Y",
        "sub": "s1_pac_sub22",
        "task": "FTT",
        "stage": "plan",
    },
    {
        "group": "O",
        "sub": "s1_pac_sub34",
        "task": "FTT",
        "stage": "go",
    },
    {
        "group": "O",
        "sub": "s1_pac_sub34",
        "task": "FTT",
        "stage": "plan",
    },
]


# ============================================================
# PATH TO MASTER DATAFRAME
# ============================================================

MASTER_PARQUET = os.path.join(
    ERPAC_DIR,
    "erpac_results.parquet"
)


# ============================================================
# CLEAN ONE CONDITION
# ============================================================

def clean_erpac_condition(
    erpac_results,
    group,
    sub,
    task,
    stage,
    roi_save_dir,
):
    """
    Remove vertices containing NaNs from ERPAC results,
    recompute ROI averages, and overwrite the condition's
    ERPAC pickle file.

    Averaging hierarchy is preserved:

        valid vertices
            ↓
        anatomical label mean
            ↓
        ROI mean across labels

    Parameters
    ----------
    erpac_results : dict
        ERPAC results for ONE subject/task/stage condition.

        Expected structure:

        erpac_results[coupling][roi][label] = {
            "erpac": array (vertices, amp_freqs, times),
            "p-values": ...,
            "times": ...,
            "vertices": ...
        }

    Returns
    -------
    clean_results : dict
        ERPAC results with bad vertices physically removed.

    rows : list of dict
        Recomputed long-format ROI ERPAC rows.

    removal_log : list of dict
        Information about removed vertices.
    """

    clean_results = {}
    rows = []
    removal_log = []

    # ========================================================
    # COUPLING LOOP
    # ========================================================

    for coupling_name, roi_dict in erpac_results.items():

        print("\n" + "=" * 80)
        print(
            f"{sub} | {task} | {stage} | "
            f"{coupling_name}"
        )
        print("=" * 80)

        clean_results[coupling_name] = {}

        # ====================================================
        # ROI LOOP
        # ====================================================

        for roi, labels in roi_dict.items():

            clean_results[coupling_name][roi] = {}

            label_means = []

            print(f"\nROI: {roi}")

            # =================================================
            # LABEL LOOP
            # =================================================

            for label_name, res in labels.items():

                erpac = np.asarray(
                    res["erpac"]
                )

                vertices = np.asarray(
                    res["vertices"]
                )

                # Shape:
                # vertices x amp_freq x time

                n_vertices_before = erpac.shape[0]

                # ------------------------------------------------
                # FIND BAD VERTICES
                # ------------------------------------------------
                # A vertex is removed if ANY point in its ERPAC
                # freq x time map is NaN or infinite.
                # ------------------------------------------------

                valid_vertex_mask = np.all(
                    np.isfinite(erpac),
                    axis=(1, 2)
                )

                bad_vertex_mask = ~valid_vertex_mask

                bad_indices = np.where(
                    bad_vertex_mask
                )[0]

                bad_vertices = vertices[
                    bad_vertex_mask
                ]

                # ------------------------------------------------
                # REMOVE BAD VERTICES
                # ------------------------------------------------

                clean_erpac = erpac[
                    valid_vertex_mask
                ]

                clean_vertices = vertices[
                    valid_vertex_mask
                ]

                # p-values need matching vertex removal
                if (
                    "p-values" in res
                    and res["p-values"] is not None
                ):

                    pvalues = np.asarray(
                        res["p-values"]
                    )

                    clean_pvalues = pvalues[
                        valid_vertex_mask
                    ]

                else:
                    clean_pvalues = None

                n_vertices_after = (
                    clean_erpac.shape[0]
                )

                n_removed = (
                    n_vertices_before
                    - n_vertices_after
                )

                # ------------------------------------------------
                # REPORT REMOVAL
                # ------------------------------------------------

                if n_removed > 0:

                    print(
                        f"  {label_name}: "
                        f"removed {n_removed} / "
                        f"{n_vertices_before} vertices"
                    )

                    print(
                        "    Removed FS vertices:",
                        bad_vertices.tolist()
                    )

                    for idx, vertex in zip(
                        bad_indices,
                        bad_vertices
                    ):

                        removal_log.append({
                            "group": group,
                            "sub": sub,
                            "task": task,
                            "stage": stage,
                            "coupling":
                                coupling_name,
                            "roi": roi,
                            "label":
                                label_name,
                            "vertex_index":
                                int(idx),
                            "vertex_number":
                                int(vertex),
                        })

                else:

                    print(
                        f"  {label_name}: "
                        f"all {n_vertices_before} "
                        f"vertices valid"
                    )

                # ------------------------------------------------
                # SAFETY CHECK
                # ------------------------------------------------

                if n_vertices_after == 0:

                    raise RuntimeError(
                        f"No valid ERPAC vertices remain for:\n"
                        f"{sub} | {task} | {stage} | "
                        f"{coupling_name} | {roi} | "
                        f"{label_name}"
                    )

                # ------------------------------------------------
                # STORE CLEAN LABEL RESULTS
                # ------------------------------------------------

                clean_results[
                    coupling_name
                ][roi][label_name] = {
                    **res,
                    "erpac":
                        clean_erpac,
                    "vertices":
                        clean_vertices,
                }

                if clean_pvalues is not None:

                    clean_results[
                        coupling_name
                    ][roi][label_name][
                        "p-values"
                    ] = clean_pvalues

                # ------------------------------------------------
                # LABEL MEAN
                # ------------------------------------------------

                label_mean = (
                    clean_erpac.mean(
                        axis=0
                    )
                )

                label_means.append(
                    label_mean
                )

            # =================================================
            # ROI MEAN
            # =================================================
            #
            # This preserves your original method:
            #
            # mean across vertices within each label
            # then mean equally across labels.
            # =================================================

            roi_mean = np.mean(
                np.stack(
                    label_means,
                    axis=0
                ),
                axis=0
            )

            # Should now contain no NaNs
            if not np.all(
                np.isfinite(roi_mean)
            ):

                raise RuntimeError(
                    f"ROI mean still contains "
                    f"non-finite values:\n"
                    f"{sub} | {task} | {stage} | "
                    f"{coupling_name} | {roi}"
                )

            # ------------------------------------------------
            # GET TIMES
            # ------------------------------------------------

            first_label = next(
                iter(
                    clean_results[
                        coupling_name
                    ][roi].values()
                )
            )

            times = np.asarray(
                first_label["times"]
            )

            # ------------------------------------------------
            # GET AMP FREQUENCIES
            # ------------------------------------------------
            #
            # If amp_freqs were stored during rerun,
            # use them.
            # Otherwise reconstruct from COUPLINGS using
            # EventRelatedPac.
            # ------------------------------------------------

            if "amp_freqs" in first_label:

                amp_freqs = np.asarray(
                    first_label["amp_freqs"]
                )

            else:

                phase_freq, amp_freq = (
                    COUPLINGS[
                        coupling_name
                    ]
                )

                p_temp = EventRelatedPac(
                    f_pha=phase_freq,
                    f_amp=amp_freq,
                )

                amp_freqs = p_temp.yvec

            # =================================================
            # CREATE LONG-FORM ROWS
            # =================================================

            for f_idx, amp_frequency in enumerate(
                amp_freqs
            ):

                for t_idx, time in enumerate(
                    times
                ):

                    rows.append({
                        "sub": sub,
                        "group": group,
                        "task": task,
                        "task_stage": stage,
                        "coupling":
                            coupling_name,
                        "roi": roi,
                        "amp_freq":
                            amp_frequency,
                        "time":
                            time,
                        "erpac_value":
                            roi_mean[
                                f_idx,
                                t_idx
                            ],
                    })

    # ========================================================
    # FINAL CHECK BEFORE SAVING
    # ========================================================

    new_df = pd.DataFrame(
        rows
    )

    n_nan = (
        new_df["erpac_value"]
        .isna()
        .sum()
    )

    print("\n" + "-" * 80)

    print(
        f"{sub} | {task} | {stage}: "
        f"{len(new_df):,} dataframe rows"
    )

    print(
        f"NaN ERPAC values after cleaning: "
        f"{n_nan}"
    )

    if n_nan > 0:

        raise RuntimeError(
            "NaNs remain after vertex removal."
        )

    # ========================================================
    # OVERWRITE ERPAC PKL
    # ========================================================

    os.makedirs(
        roi_save_dir,
        exist_ok=True
    )

    pkl_path = os.path.join(
        roi_save_dir,
        f"{sub}_{task}_{stage}_erpac_results.pkl"
    )

    with open(
        pkl_path,
        "wb"
    ) as f:

        pickle.dump(
            clean_results,
            f
        )

    print(
        f"Saved cleaned ERPAC pickle:\n"
        f"{pkl_path}"
    )

    return (
        clean_results,
        rows,
        removal_log,
    )

In [40]:
# ============================================================
# CLEAN ALL AFFECTED CONDITIONS
# ============================================================

replacement_rows = []
all_removed_vertices = []


for item in AFFECTED_CONDITIONS:

    group = item["group"]
    sub = item["sub"]
    task = item["task"]
    stage = item["stage"]

    condition_key = (
        group,
        sub,
        task,
        stage,
    )

    print("\n\n")
    print("#" * 100)
    print(
        f"CLEANING {group} | {sub} | "
        f"{task} | {stage}"
    )
    print("#" * 100)

    # ERPAC from your rerun diagnostic
    erpac_results_condition = (
        rerun_erpac_results[
            condition_key
        ]
    )

    # Same directory structure you used previously
    roi_save_dir = os.path.join(
        ROI_STCS_DIR,
        group,
        task,
        stage
    )

    (
        clean_results,
        new_rows,
        removed_vertices,
    ) = clean_erpac_condition(
        erpac_results=
            erpac_results_condition,
        group=group,
        sub=sub,
        task=task,
        stage=stage,
        roi_save_dir=
            roi_save_dir,
    )

    replacement_rows.extend(
        new_rows
    )

    all_removed_vertices.extend(
        removed_vertices
    )




####################################################################################################
CLEANING Y | s1_pac_sub22 | FTT | go
####################################################################################################

s1_pac_sub22 | FTT | go | theta_gamma

ROI: M1
  G_precentral-lh: removed 1 / 60 vertices
    Removed FS vertices: [1457]

ROI: S1
  G_postcentral-lh: all 55 vertices valid
  S_postcentral-lh: all 69 vertices valid

ROI: PMC
  S_precentral-sup-part-lh: all 28 vertices valid
  S_precentral-inf-part-lh: all 31 vertices valid
  G_front_inf-Opercular-lh: all 25 vertices valid

ROI: SMA
  G_and_S_paracentral-lh: all 35 vertices valid
  G_front_sup-lh: all 134 vertices valid

s1_pac_sub22 | FTT | go | alpha_gamma

ROI: M1
  G_precentral-lh: removed 1 / 60 vertices
    Removed FS vertices: [1457]

ROI: S1
  G_postcentral-lh: all 55 vertices valid
  S_postcentral-lh: all 69 vertices valid

ROI: PMC
  S_precentral-sup-part-lh: all 28 vertices valid
  S_pre

In [41]:
removed_vertices_df = pd.DataFrame(
    all_removed_vertices
)

print("\n\nREMOVED VERTICES")
print("=" * 100)

print(
    removed_vertices_df
    .drop_duplicates(
        subset=[
            "sub",
            "roi",
            "label",
            "vertex_number",
        ]
    )
    .to_string(
        index=False
    )
)



REMOVED VERTICES
group          sub task stage    coupling roi                    label  vertex_index  vertex_number
    Y s1_pac_sub22  FTT    go theta_gamma  M1          G_precentral-lh            44           1457
    O s1_pac_sub34  FTT    go theta_gamma PMC S_precentral-inf-part-lh            13           1002
    O s1_pac_sub34  FTT    go theta_gamma SMA           G_front_sup-lh           112           2102


In [42]:
replacement_df = pd.DataFrame(
    replacement_rows
)

print(
    "Replacement rows:",
    replacement_df.shape
)

print(
    "NaNs in replacement rows:",
    replacement_df[
        "erpac_value"
    ].isna().sum()
)

Replacement rows: (1082160, 9)
NaNs in replacement rows: 0


In [43]:
# ============================================================
# IDENTIFY OLD ROWS TO REMOVE
# ============================================================

remove_mask = np.zeros(
    len(erpac_df),
    dtype=bool
)


for item in AFFECTED_CONDITIONS:

    mask = (
        (erpac_df["group"]
         == item["group"])
        &
        (erpac_df["sub"]
         == item["sub"])
        &
        (erpac_df["task"]
         == item["task"])
        &
        (erpac_df["task_stage"]
         == item["stage"])
    )

    print(
        f"{item['sub']} | "
        f"{item['task']} | "
        f"{item['stage']}: "
        f"removing {mask.sum():,} old rows"
    )

    remove_mask |= mask

s1_pac_sub22 | FTT | go: removing 270,540 old rows
s1_pac_sub22 | FTT | plan: removing 270,540 old rows
s1_pac_sub34 | FTT | go: removing 270,540 old rows
s1_pac_sub34 | FTT | plan: removing 270,540 old rows


In [44]:
erpac_df_clean = (
    erpac_df.loc[
        ~remove_mask
    ]
    .copy()
)

In [45]:
# ============================================================
# APPEND CLEAN REPLACEMENT ROWS
# ============================================================

erpac_df_clean = pd.concat(
    [
        erpac_df_clean,
        replacement_df,
    ],
    ignore_index=True,
)

In [46]:
erpac_df_clean = (
    erpac_df_clean
    .sort_values(
        [
            "group",
            "sub",
            "task",
            "task_stage",
            "coupling",
            "roi",
            "amp_freq",
            "time",
        ]
    )
    .reset_index(
        drop=True
    )
)

In [47]:
erpac_df_clean

,sub,group,task,task_stage,coupling,roi,amp_freq,time,erpac_value
0,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.500,0.136649
1,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.498,0.126122
2,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.496,0.114843
3,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.494,0.110220
4,s1_pac_sub12,O,FTT,go,alpha_gamma,M1,32.5,-0.492,0.107816
...,...,...,...,...,...,...,...,...,...
25430755,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.492,0.153508
25430756,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.494,0.151223
25430757,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.496,0.145920
25430758,s1_pac_sub77,Y,FTT,plan,theta_gamma,SMA,76.5,0.498,0.146189


In [48]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print("\nFINAL DATAFRAME")
print("=" * 100)

print(
    "Shape:",
    erpac_df_clean.shape
)

print(
    "Total NaNs:",
    erpac_df_clean[
        "erpac_value"
    ].isna().sum()
)


# Specifically inspect affected conditions
check_cleaned = (
    erpac_df_clean[
        erpac_df_clean["sub"].isin(
            [
                "s1_pac_sub22",
                "s1_pac_sub34",
            ]
        )
    ]
    .groupby(
        [
            "group",
            "sub",
            "task",
            "task_stage",
            "coupling",
            "roi",
        ]
    )
    ["erpac_value"]
    .agg(
        n="size",
        n_nan=lambda x:
            x.isna().sum(),
    )
    .reset_index()
)

check_cleaned["pct_nan"] = (
    100
    * check_cleaned["n_nan"]
    / check_cleaned["n"]
)

print(
    check_cleaned.to_string(
        index=False
    )
)


FINAL DATAFRAME
Shape: (25430760, 9)
Total NaNs: 0
group          sub task task_stage    coupling roi     n  n_nan  pct_nan
    O s1_pac_sub34  FTT         go alpha_gamma  M1 22545      0      0.0
    O s1_pac_sub34  FTT         go alpha_gamma PMC 22545      0      0.0
    O s1_pac_sub34  FTT         go alpha_gamma  S1 22545      0      0.0
    O s1_pac_sub34  FTT         go alpha_gamma SMA 22545      0      0.0
    O s1_pac_sub34  FTT         go  beta_gamma  M1 22545      0      0.0
    O s1_pac_sub34  FTT         go  beta_gamma PMC 22545      0      0.0
    O s1_pac_sub34  FTT         go  beta_gamma  S1 22545      0      0.0
    O s1_pac_sub34  FTT         go  beta_gamma SMA 22545      0      0.0
    O s1_pac_sub34  FTT         go theta_gamma  M1 22545      0      0.0
    O s1_pac_sub34  FTT         go theta_gamma PMC 22545      0      0.0
    O s1_pac_sub34  FTT         go theta_gamma  S1 22545      0      0.0
    O s1_pac_sub34  FTT         go theta_gamma SMA 22545      0      0.0

In [49]:
# ============================================================
# SAVE UPDATED MASTER DATASET
# ============================================================

erpac_df_clean.to_parquet(
    MASTER_PARQUET,
    engine="pyarrow",
    compression="zstd",
    index=False,
)

print(
    f"\nUpdated master ERPAC dataframe saved:\n"
    f"{MASTER_PARQUET}"
)


Updated master ERPAC dataframe saved:
F:\# study 2\eeg_data\erpac\erpac_results.parquet
